In [1]:
import sys
from pathlib import Path
here = Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "src" / "model.py").exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
from scipy.stats import loguniform, randint
from sklearn.model_selection import TimeSeriesSplit, cross_validate, RandomizedSearchCV
from src.data import ORDERS_PATH, TARGET
from src.model import FEATURES, time_split, make_logistic, make_forest, make_boosting, score

orders = pd.read_csv(ORDERS_PATH, parse_dates=["order_date"])
train, test = time_split(orders)
cv = TimeSeriesSplit(n_splits=5)

In [2]:
rows = []
for name, make in [("logistic regression", make_logistic),
                   ("random forest", make_forest),
                   ("gradient boosting", make_boosting)]:
    out = cross_validate(make(), train[FEATURES], train[TARGET], cv=cv,
                         scoring=["roc_auc", "accuracy"])
    rows.append({"model": name,
                 "auc_mean": round(out["test_roc_auc"].mean(), 4),
                 "auc_std": round(out["test_roc_auc"].std(), 4),
                 "acc_mean": round(out["test_accuracy"].mean(), 4),
                 "fit_seconds": round(out["fit_time"].mean(), 1)})
pd.DataFrame(rows)

,model,auc_mean,auc_std,acc_mean,fit_seconds
0,logistic regression,0.7463,0.0037,0.7024,0.4
1,random forest,0.7730,0.0046,0.7219,1.1
2,gradient boosting,0.7762,0.0029,0.7157,0.9


In [3]:
search = RandomizedSearchCV(
    make_boosting(),
    param_distributions={
        "model__learning_rate":     loguniform(0.01, 0.3),
        "model__max_iter":          randint(100, 600),
        "model__max_leaf_nodes":    randint(8, 64),
        "model__min_samples_leaf":  randint(20, 200),
        "model__l2_regularization": loguniform(1e-3, 10),
    },
    n_iter=20, cv=cv, scoring="roc_auc", random_state=42, n_jobs=-1)
search.fit(train[FEATURES], train[TARGET])
print("best CV AUC:", round(search.best_score_, 4))
print(search.best_params_)

best CV AUC: 0.7762
{'model__l2_regularization': np.float64(0.02161894240657444), 'model__learning_rate': np.float64(0.016256670876862694), 'model__max_iter': 230, 'model__max_leaf_nodes': 29, 'model__min_samples_leaf': 72}


In [4]:
def rules_prob(df):
    """No ML: the two rules + the late rate implied by the transit table (4 of 5 days, 2 of 5 days)."""
    p = df["shipping_mode"].map({"First Class": 1.0, "Second Class": 0.8,
                                 "Standard Class": 0.4, "Same Day": 0.0})
    after_noon = (df["shipping_mode"] == "Same Day") & (df["order_hour"] >= 12)
    return p.where(~after_noon, 1.0)

default = make_boosting().fit(train[FEATURES], train[TARGET])
final = [
    score("gradient boosting (default)", test[TARGET], default.predict_proba(test[FEATURES])[:, 1]),
    score("gradient boosting (tuned)",   test[TARGET], search.best_estimator_.predict_proba(test[FEATURES])[:, 1]),
    score("2 rules, no ML",              test[TARGET], rules_prob(test)),
]
pd.DataFrame(final)

,model,accuracy,roc_auc
0,gradient boosting (default),0.7223,0.7773
1,gradient boosting (tuned),0.7239,0.7764
2,"2 rules, no ML",0.7250,0.7758
